### Preparing data and writing to gold layer ###

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
 gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/gold/fact_order_items/"

In [0]:
df_silver_orders = spark.readStream.format("delta").option("readChangeFeed", "true").table(f"{catalog_name}.silver.slv_order_items")
#display(df_silver_orders, checkpointLocation = gold_checkpoint_path, numrows = 10)

In [0]:
df_gold_orders = df_silver_orders.filter("_change_type IN('updateAfter','insert')")

In [0]:
df_gold_orders = df_gold_orders.withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
df_gold_orders = df_gold_orders.withColumn("discount_amount", F.col("gross_amount") * F.ceil(F.col("discount_pct")/100))
df_gold_orders = df_gold_orders.withColumn("sales_amount", F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount"))
df_gold_orders = df_gold_orders.withColumn("date_id", F.date_format(F.col("dt"),"yyyyMMdd").cast("int"))
df_gold_orders = df_gold_orders.withColumn("coupon_flag", F.when(F.col("coupon_code").isNotNull(),F.lit(1)).otherwise(F.lit(0)))

In [0]:
df_gold_orders = df_gold_orders.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sales_amount").alias("net_amount")
)

In [0]:
def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.transaction_id = batch_table.transaction_id AND gold_table.seq_no = batch_table.seq_no",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
df_gold_orders.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).format("delta").option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()

In [0]:
#spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()

In [0]:
#spark.sql(f"SELECT max(transaction_date) FROM {catalog_name}.gold.gld_fact_order_items").show()